# Chickpea Disease Detection — Hybrid CNN-RNN
### Final Year Project Demonstration Notebook
> *Jai Shriram Engineering College, Tirupur*

This notebook demonstrates the complete pipeline:
1. Synthetic dataset generation
2. Image preprocessing (Algorithm 1 from paper)
3. Model architecture overview
4. Training
5. Evaluation and visualizations

In [ ]:
import sys, os
sys.path.insert(0, '../backend')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import cv2
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

## 1. Generate Synthetic Dataset

In [ ]:
from dataset_generator import build_dataset, GENERATORS, CLASSES

# Generate 80 images per class
meta = build_dataset('../dataset', n_per_class=80)
print('\nClass mapping:', meta['class_to_idx'])

In [ ]:
# Visualise one sample from each class
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('Sample Leaf Images per Disease Class', fontsize=14, fontweight='bold')

for ax, cls in zip(axes, CLASSES):
    img = GENERATORS[cls]()
    ax.imshow(img)
    ax.set_title(cls, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../models/sample_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → models/sample_classes.png')

## 2. Image Preprocessing Pipeline (Algorithm 1)

In [ ]:
from preprocessing import (
    pil_to_cv2, cv2_to_pil,
    remove_noise, enhance_contrast, segment_leaf, normalize_image
)

# Load a sample blight image
from dataset_generator import generate_blight
pil_img   = generate_blight()
bgr_img   = pil_to_cv2(pil_img)

# Run each step
denoised   = remove_noise(bgr_img)
enhanced   = enhance_contrast(denoised)
segmented, mask = segment_leaf(enhanced)
normalized = normalize_image(segmented)

steps = [
    ('1. Original',   bgr_img),
    ('2. Denoised\n(Gaussian)',  denoised),
    ('3. Enhanced\n(CLAHE)',     enhanced),
    ('4. Segmented\n(Otsu)',     segmented),
    ('5. Normalized\n[0,1]',    normalized),
]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Algorithm 1 — Preprocessing Pipeline (Blight Sample)', fontsize=13, fontweight='bold')

for ax, (title, img) in zip(axes, steps):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../models/preprocessing_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Architecture

In [ ]:
from model import get_model, DISEASE_CLASSES

model = get_model(num_classes=4, image_size=128)
print(model)

total  = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters    : {total:,}')
print(f'Trainable parameters: {train_:,}')

In [ ]:
# Test forward pass
dummy  = torch.randn(2, 3, 128, 128)
output = model(dummy)
print(f'Input shape  : {dummy.shape}')
print(f'Output shape : {output.shape}  ← (batch, num_classes)')
print(f'Classes      : {list(DISEASE_CLASSES.values())}')

## 4. Train the Model

In [ ]:
# Run training (adjust epochs as needed)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../backend/train.py',
     '--data_dir', '../dataset',
     '--model_dir', '../models',
     '--epochs', '20',
     '--batch_size', '16'],
    capture_output=False
)
print('Return code:', result.returncode)

## 5. Evaluate & Visualise

In [ ]:
import json

with open('../models/metrics.json') as f:
    metrics = json.load(f)

print('='*45)
print('         TEST SET PERFORMANCE')
print('='*45)
print(f"Accuracy  : {metrics['test_accuracy']:.4f}")
print(f"Precision : {metrics['test_precision']:.4f}")
print(f"Recall    : {metrics['test_recall']:.4f}")
print(f"F1-Score  : {metrics['test_f1']:.4f}")
print('\nPer-class results:')
for cls, scores in metrics.get('per_class', {}).items():
    print(f"  {cls:10s}: F1={scores['f1-score']:.3f}  Prec={scores['precision']:.3f}  Rec={scores['recall']:.3f}")

In [ ]:
# Show training curves
img_path = '../models/plots/training_curves.png'
if os.path.exists(img_path):
    from IPython.display import Image as IPImage
    display(IPImage(img_path))

In [ ]:
# Show confusion matrix
img_path = '../models/plots/confusion_matrix.png'
if os.path.exists(img_path):
    from IPython.display import Image as IPImage
    display(IPImage(img_path))

## 6. Single Image Inference Demo

In [ ]:
import torch.nn.functional as F
from preprocessing import preprocess_for_inference
import io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load trained model
ckpt  = torch.load('../models/best_model.pth', map_location=device)
model = get_model(num_classes=4, image_size=128)
model.load_state_dict(ckpt['model_state'])
model.eval().to(device)

# Generate a test rust image
from dataset_generator import generate_rust
test_pil = generate_rust()

# Convert to bytes
buf = io.BytesIO()
test_pil.save(buf, format='PNG')
image_bytes = buf.getvalue()

# Inference
tensor, pil_proc, pil_orig = preprocess_for_inference(image_bytes)
tensor = tensor.to(device)

with torch.no_grad():
    logits = model(tensor)
    probs  = F.softmax(logits, dim=1)[0]
    pred   = probs.argmax().item()

print(f'True class  : Rust')
print(f'Prediction  : {DISEASE_CLASSES[pred]}')
print(f'Confidence  : {probs[pred].item()*100:.1f}%')
print('\nAll probabilities:')
for i, p in enumerate(probs):
    bar = '█' * int(p.item() * 30)
    print(f'  {DISEASE_CLASSES[i]:8s} | {bar:<30} {p.item()*100:.1f}%')

In [ ]:
# Show original vs preprocessed
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(pil_orig); ax1.set_title('Original'); ax1.axis('off')
ax2.imshow(pil_proc); ax2.set_title('Preprocessed'); ax2.axis('off')
plt.suptitle(f'Inference: {DISEASE_CLASSES[pred]} ({probs[pred].item()*100:.1f}%)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()